In [ ]:
import pandas as pd

data = {
    "Question": [
        "What is your menu?",
        "Do you offer delivery?",
        "Can I book a table?",
        "What are your timings?",
        "Where are you located?",
        "How can I track my order?"
    ],
    "Answer": [
        "We offer pizza, burger, pasta, and shawarma.",
        "Yes, we provide home delivery.",
        "Yes, you can reserve a table anytime.",
        "We are open from 10 AM to 11 PM.",
        "We are located in the city center.",
        "You can track your order using item name."
    ]
}

restaurant_df = pd.DataFrame(data)
restaurant_df

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

restaurant_df["Cleaned_Question"] = restaurant_df["Question"].apply(clean_text)
restaurant_df

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

texts = restaurant_df["Cleaned_Question"].astype(str).tolist()

embeddings = model.encode(texts)

embeddings = np.array(embeddings).astype("float32")

In [ ]:
np.save('restaurant_embeddings.npy', embeddings)

embeddings = np.load('restaurant_embeddings.npy')

In [ ]:
pip install faiss-cpu

In [ ]:
import faiss

dimensions = embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimensions)
faiss_index.add(embeddings)

faiss.write_index(faiss_index, 'restaurant_index.index')

In [ ]:
def get_similar_answer(query, count=3):
    query = clean_text(query)

    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distance, indices = faiss_index.search(query_embedding, count)

    for i in range(count):
        print(f"Result {i+1}, Distance: {distance[0][i]}")
        print("Q:", restaurant_df['Question'].iloc[indices[0][i]])
        print("A:", restaurant_df['Answer'].iloc[indices[0][i]])
        print("------")

In [ ]:
get_similar_answer("Do you have food?")

In [ ]:
def chatbot(query):
    query = clean_text(query)

    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distance, indices = faiss_index.search(query_embedding, 1)

    best_index = indices[0][0]
    best_distance = distance[0][0]

    if best_distance > 1.5:
        return "Sorry, I don't understand."

    return restaurant_df['Answer'].iloc[best_index]

In [ ]:
print(chatbot("Book a table"))
print(chatbot("Do you deliver?"))
print(chatbot("Where is your restaurant?"))